# ASG Airlines 

ingest raw operational flight data (flights, bookings, payments, passengers),
detect and fix data-quality issues, mask PII, model an analytics-ready fact table, and
compute the business KPIs requested in the case study.


In [ ]:
import pandas as pd
import numpy as np
import hashlib
import re
import logging
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("asg_pipeline")

SRC = "UseCase - Airlines.xlsx"  
RAW_DIR, CLEAN_DIR, OUT_DIR = "raw", "cleaned", "output"
for d in (RAW_DIR, CLEAN_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)


SALT = os.environ.get("ASG_PII_SALT", "ASG_AIRLINES_2026_SALT")

## 1. Data Ingestion



In [40]:
def ingest(path=None):

    if path is None:
        path = SRC

    log.info("Ingesting raw sheets from %s", path)

    sheets = pd.read_excel(path, sheet_name=None)

    for name, df in sheets.items():

        out_path = f"{RAW_DIR}/{name}_raw.csv"

        df.to_csv(out_path, index=False)

        log.info(
            "  -> %-12s %5d rows x %d cols -> %s",
            name,
            len(df),
            df.shape[1],
            out_path
        )

    return sheets

In [41]:
from pathlib import Path

SRC = Path(r"C:\Users\nvnir\OneDrive\Desktop\use case\UseCase - Airlines.xlsx")

print("Path:", SRC)
print("Exists:", SRC.exists())

Path: C:\Users\nvnir\OneDrive\Desktop\use case\UseCase - Airlines.xlsx
Exists: True


In [42]:
sheets = ingest()

INFO | Ingesting raw sheets from C:\Users\nvnir\OneDrive\Desktop\use case\UseCase - Airlines.xlsx
INFO |   -> flights       1020 rows x 7 cols -> raw/flights_raw.csv
INFO |   -> payments      1000 rows x 4 cols -> raw/payments_raw.csv
INFO |   -> bookings      1000 rows x 9 cols -> raw/bookings_raw.csv
INFO |   -> passengers    1039 rows x 9 cols -> raw/passengers_raw.csv


In [43]:
print(sheets.keys())

dict_keys(['flights', 'payments', 'bookings', 'passengers'])


## 2. PII Masking Helpers

The passengers and bookings sheets carry real passenger PII: name, email, phone,
Aadhaar ID, date of birth, passport number, emergency contact. Per the case-study
requirement, none of this can flow into the analytics/reporting layer in the clear.

Strategy:
- **Irreversible hash (SHA-256 + salt)** for identifiers we only need to join on or
  de-duplicate by (Aadhaar, passport, passenger_id -> passenger_key). A hash lets us
  count/group without ever exposing or reconstructing the original value.
- **Partial masking** for fields an operations user might legitimately need to see a
  fragment of (email, phone).
- **Generalization** for date of birth → keep only birth year, which is enough for
  age-band analytics but not enough to identify someone.
- Full name, exact phone/email, Aadhaar/passport numbers, and emergency-contact details
  are dropped entirely from every file that leaves the secured zone 

In [44]:
def sha256_hash(value):
    if pd.isna(value):
        return None
    return hashlib.sha256((str(value) + SALT).encode()).hexdigest()[:16]

def mask_email(email):
    if pd.isna(email) or "@" not in str(email):
        return None
    local, domain = str(email).split("@", 1)
    masked_local = local[0] + "*" if len(local) <= 2 else local[0] + "*" * (len(local) - 2) + local[-1]
    return f"{masked_local}@{domain}"

def mask_phone(phone):
    if pd.isna(phone):
        return None
    digits = re.sub(r"\D", "", str(phone))
    return "*" * len(digits) if len(digits) < 4 else "XXXXXX" + digits[-4:]

## 3. Cleaning — `flights`

Issues found in the profiling step and how each is handled:

| Issue | Rule applied |
|---|---|
| `flight_id` may be corrupted | Validate against the `AA000` pattern (2 letters/digits + 3 digits); none failed in this extract, but the check runs on every load |
| 15 exact duplicate rows | Dropped |
| 41 missing + several literal `"UNKNOWN"` `airline` values | Repaired using the 1:1 mapping between the flight-number **prefix** (`6F`→IndiGo, `AI`→Air India, `SJ`→SpiceJet, `UK`→Vistara) and airline — recovers 69 of the affected rows; anything left is labelled `"Unknown"` |
| Inconsistent station-code casing/whitespace | Upper-cased and trimmed; a `route` column (`SOURCE-DEST`) is derived |
| Corrupted / illogical arrival dates (e.g. arrival stamped a day *before* departure) | If `arrival <= departure`, re-anchor the arrival date onto the departure date first (fixes a bad date digit); if it's still not after departure, treat it as a genuine **overnight (cross-day)** flight and add one day |
| Cross-day / overnight flights | Handled naturally once arrival is a full datetime — flagged with `is_overnight` for reporting |
| Unrealistic duration outliers | Recomputed `duration_minutes` from the corrected timestamps (not the untrustworthy original `duration` field) and flagged anything `<20 min` or `>360 min` as `is_anomaly` for the Delay/Anomaly Insights page, rather than silently dropping it |


In [45]:
FLIGHT_ID_RE = re.compile(r"^[A-Z0-9]{2}\d{3}$")
PREFIX_TO_AIRLINE = {"6F": "IndiGo", "AI": "Air India", "SJ": "SpiceJet", "UK": "Vistara"}

def clean_flights(df):
    df = df.copy()
    n0 = len(df)

    df["flight_id"] = df["flight_id"].astype(str).str.strip().str.upper()
    df["is_valid_flight_id"] = df["flight_id"].str.match(FLIGHT_ID_RE)
    log.info("flights: %d/%d flight_id values fail the AA000 pattern check", (~df['is_valid_flight_id']).sum(), n0)

    dup_count = df.duplicated().sum()
    df = df.drop_duplicates()
    log.info("flights: dropped %d exact duplicate rows", dup_count)

    df["prefix"] = df["flight_id"].str[:2]
    df["airline"] = df["airline"].replace({"UNKNOWN": np.nan})
    missing_before = df["airline"].isna().sum()
    df["airline"] = df.apply(
        lambda r: PREFIX_TO_AIRLINE.get(r["prefix"], r["airline"]) if pd.isna(r["airline"]) else r["airline"],
        axis=1)
    df["airline"] = df["airline"].fillna("Unknown")
    log.info("flights: repaired %d missing/UNKNOWN airline values from the flight-number prefix", missing_before)

    df["source"] = df["source"].astype(str).str.strip().str.upper()
    df["destination"] = df["destination"].astype(str).str.strip().str.upper()
    df["route"] = df["source"] + "-" + df["destination"]

    df["departure_time"] = pd.to_datetime(df["departure_time"], errors="coerce")
    df["arrival_time"] = pd.to_datetime(df["arrival_time"], errors="coerce")

    def fix_arrival(row):
        dep, arr = row["departure_time"], row["arrival_time"]
        if pd.isna(dep) or pd.isna(arr):
            return arr, False
        corrected = False
        if arr <= dep:
            same_day_arr = arr.replace(year=dep.year, month=dep.month, day=dep.day)
            arr = same_day_arr if same_day_arr > dep else same_day_arr + pd.Timedelta(days=1)
            corrected = True
        return arr, corrected

    fixed = df.apply(fix_arrival, axis=1, result_type="expand")
    df["arrival_time"], df["date_corrected"] = fixed[0], fixed[1]
    log.info("flights: corrected %d record(s) with corrupted/illogical arrival dates", df["date_corrected"].sum())

    df["duration_minutes"] = (df["arrival_time"] - df["departure_time"]).dt.total_seconds() / 60
    df["is_overnight"] = df["arrival_time"].dt.date > df["departure_time"].dt.date
    df["is_anomaly"] = (df["duration_minutes"] < 20) | (df["duration_minutes"] > 360)

    df = df.drop(columns=["prefix"])
    log.info("flights: clean shape = %s", df.shape)
    return df

flights_c = clean_flights(sheets["flights"])
flights_c.head()

INFO | flights: 0/1020 flight_id values fail the AA000 pattern check
INFO | flights: dropped 15 exact duplicate rows
INFO | flights: repaired 69 missing/UNKNOWN airline values from the flight-number prefix
INFO | flights: corrected 1 record(s) with corrupted/illogical arrival dates
INFO | flights: clean shape = (1005, 13)


,flight_id,airline,source,destination,departure_time,arrival_time,duration,is_valid_flight_id,route,date_corrected,duration_minutes,is_overnight,is_anomaly
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00,True,CCU-MAA,False,174.0,True,False
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00,True,BOM-CCU,False,108.0,True,False
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00,True,BOM-CCU,False,105.0,True,False
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00,True,BOM-CCU,False,156.0,True,False
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00,True,MAA-BOM,False,299.0,True,False


## 4. Cleaning — `passengers` (PII masked)

- 39 rows shared a duplicate `passenger_id` — kept the first occurrence.
- 10 missing `last_name` → labelled `"Unknown"` (not enough signal to infer it).
- Every direct identifier is hashed or masked; `birth_year` replaces full DOB.

In [46]:
def clean_passengers(df):
    df = df.copy()
    dup = df["passenger_id"].duplicated().sum()
    df = df.drop_duplicates(subset="passenger_id", keep="first")
    log.info("passengers: dropped %d duplicate passenger_id rows", dup)

    df["last_name"] = df["last_name"].fillna("Unknown")
    df["date_of_birth"] = pd.to_datetime(df["date_of_birth"], errors="coerce")

    df["passenger_key"] = df["passenger_id"].apply(sha256_hash)
    df["email_masked"] = df["email"].apply(mask_email)
    df["phone_masked"] = df["phone"].apply(mask_phone)
    df["aadhaar_hash"] = df["aadhaar_id"].apply(sha256_hash)
    df["birth_year"] = df["date_of_birth"].dt.year
    return df

passengers_c = clean_passengers(sheets["passengers"])
passengers_c[["passenger_id", "passenger_key", "email_masked", "phone_masked", "aadhaar_hash", "age", "gender", "birth_year"]].head()

INFO | passengers: dropped 39 duplicate passenger_id rows


,passenger_id,passenger_key,email_masked,phone_masked,aadhaar_hash,age,gender,birth_year
0,P1000,dc2c68ea37121feb,v***************e@gmail.com,XXXXXX3790,c6ee7353091bdb46,52,F,1974
1,P1001,69117f49cca0da1f,k***********y@hotmail.com,XXXXXX2297,c6ca6c1ed50036fb,15,M,2011
2,P1002,4ba6860d4c955524,m********u@outlook.com,XXXXXX5092,1cc74e389fd54759,72,M,1954
3,P1003,a514b7ddf9feaaf7,m*********a@hotmail.com,XXXXXX7151,cdb8e802426fba2e,61,F,1965
4,P1004,f632a7a5306c3287,s*************e@outlook.com,XXXXXX5113,fdf1b874e13b9fa5,21,M,2005


## 5. Cleaning — `bookings` (PII masked)

- 45 missing + several literal `"INVALID"` `status` values → normalised to `"UNKNOWN"`
  (kept as its own category rather than guessed, since a wrong guess here would distort
  the cancellation-rate KPI).
- `passport_number` is hashed; `emergency_contact_phone` is masked; `emergency_contact_name`
  is dropped outright — it has no analytical value and is pure PII risk.

In [47]:
def clean_bookings(df):
    df = df.copy()
    df["status"] = df["status"].replace({"INVALID": np.nan}).fillna("UNKNOWN")
    df["passport_hash"] = df["passport_number"].apply(sha256_hash)
    df["emergency_contact_phone_masked"] = df["emergency_contact_phone"].apply(mask_phone)
    df = df.drop(columns=["passport_number", "emergency_contact_name", "emergency_contact_phone"])
    return df

bookings_c = clean_bookings(sheets["bookings"])
bookings_c.head()

,booking_id,passenger_id,flight_id,booking_date,status,seat_number,passport_hash,emergency_contact_phone_masked
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,3D,50cb38b0cfcc8496,XXXXXX5128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,18A,e6a00ecf8549349c,XXXXXX8662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,11e59a6a94bd8174,XXXXXX8220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,33A,befcdb739ed6246e,XXXXXX6839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,25C,c1d412afedd9dca0,XXXXXX8266


## 6. Cleaning — `payments`

- `amount` contained **literal `"INVALID"` strings** mixed in with real numbers — a
  classic schema-inconsistency trap. `pd.to_numeric(..., errors="coerce")` turns both
  that and the true NaNs into a single missing-value signal, which we then impute with
  the **median amount for that payment method** (more realistic than a single global
  median, since UPI/CARD/NETBANKING transactions cluster differently).
- Several bookings have **more than one payment row** (retries/instalments) — handled
  in the modelling step below rather than here, so we don't lose that signal.

In [48]:
def clean_payments(df):
    df = df.copy()
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["amount_was_imputed"] = df["amount"].isna()
    missing = df["amount"].isna().sum()
    median_by_method = df.groupby("payment_method")["amount"].transform("median")
    df["amount"] = df["amount"].fillna(median_by_method)
    log.info("payments: found invalid/non-numeric + missing amount values (%d total); imputed with the payment-method median", missing)
    return df

payments_c = clean_payments(sheets["payments"])
payments_c.head()

INFO | payments: found invalid/non-numeric + missing amount values (78 total); imputed with the payment-method median


,payment_id,booking_id,amount,payment_method,amount_was_imputed
0,PAY1000,B1116,9883.49,NETBANKING,False
1,PAY1001,B1738,8457.96,NETBANKING,False
2,PAY1002,B1873,6495.37,UPI,False
3,PAY1003,B1914,5079.38,NETBANKING,False
4,PAY1004,B1967,12518.31,CARD,False


## 7. Data Modelling — building the analytics-ready fact table

Two join subtleties surfaced while modelling, both documented as explicit assumptions:

1. **`flight_id` is a flight *number*, not a unique key** — the same number operates on
   many different days, and `bookings` doesn't carry a flight date to disambiguate which
   occurrence a booking used. *Assumption:* keep the first occurrence of each `flight_id`
   as its representative record, so flight attributes (route, airline, duration) are still
   usable for reporting even though we can't pin an exact date per booking.
2. **`payments` can have multiple rows per `booking_id`** (retries/instalments) — a naive
   join fans a 1,000-row bookings table out to 1,366+ rows. *Assumption:* the amount
   actually collected for a booking is the **sum** of its payment rows; the attempt count
   is kept as a data-quality/ops signal (`payment_attempts > 1` is worth investigating).

The result is a single **fact table at booking grain** (1 row = 1 booking) — this is the
table Power BI will connect to. In Azure this step is a **Databricks/Synapse notebook
writing Delta tables into the Silver → Gold layers** of the lakehouse.

In [49]:
def build_fact_bookings(flights, bookings, payments, passengers):
    flights_dim = flights.drop_duplicates(subset="flight_id", keep="first")

    pay_agg = (
        payments.groupby("booking_id")
        .agg(amount=("amount", "sum"),
             payment_attempts=("payment_id", "count"),
             payment_method=("payment_method", "last"))
        .reset_index()
    )

    fact = (
        bookings
        .merge(flights_dim, on="flight_id", how="left", suffixes=("", "_flight"))
        .merge(pay_agg, on="booking_id", how="left")
        .merge(passengers[["passenger_id", "passenger_key", "age", "gender", "birth_year"]],
               on="passenger_id", how="left")
    )
    keep = ["booking_id", "passenger_key", "flight_id", "airline", "route", "source", "destination",
            "departure_time", "arrival_time", "duration_minutes", "is_overnight", "is_anomaly",
            "booking_date", "status", "amount", "payment_attempts", "payment_method",
            "age", "gender", "birth_year"]
    assert len(fact) == len(bookings), "fact table must stay at booking grain (1 row = 1 booking)"
    return fact[keep]

fact = build_fact_bookings(flights_c, bookings_c, payments_c, passengers_c)
print("fact_bookings shape:", fact.shape)
fact.head()

fact_bookings shape: (1000, 20)


,booking_id,passenger_key,flight_id,airline,route,source,destination,departure_time,arrival_time,duration_minutes,is_overnight,is_anomaly,booking_date,status,amount,payment_attempts,payment_method,age,gender,birth_year
0,B1000,b7a33f63db7abfa1,AI192,Air India,MAA-BOM,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,299.0,True,False,2025-06-14 11:37:36.951,CANCELLED,NaN,NaN,NaN,4,M,2022
1,B1001,a9b279b0299cec07,6F026,IndiGo,BOM-CCU,BOM,CCU,2026-04-20 22:46:42.000,2026-04-21 03:13:42.000,267.0,True,False,2025-11-02 11:37:36.951,CANCELLED,NaN,NaN,NaN,82,F,1944
2,B1002,9fb5f23aa8f77b5b,SJ010,SpiceJet,CCU-MAA,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,174.0,True,False,2025-08-25 11:37:36.951,CANCELLED,8593.08,2.0,NETBANKING,62,F,1964
3,B1003,fe1860e5bf8099b9,AI069,Air India,DEL-BOM,DEL,BOM,2026-04-20 22:43:41.702,2026-04-21 02:29:41.702,226.0,True,False,2025-12-30 11:37:36.951,CONFIRMED,NaN,NaN,NaN,60,M,1966
4,B1004,b76e6dbd79df9425,UK003,Vistara,HYD-DEL,HYD,DEL,2026-04-20 21:41:41.700,2026-04-21 02:19:41.700,278.0,True,False,2025-10-02 11:37:36.951,PENDING,12222.48,1.0,UPI,33,F,1993


## 8. Business KPIs

Flight-level KPIs (duration, traffic, anomalies, airline mix) are computed from the
**flight dimension** (every scheduled flight instance). Money/booking-level KPIs
(revenue, cancellation rate, fares) are computed from the **fact table** (booking grain).
Anomalous flights are excluded from the duration *average* so a couple of bad records
don't distort it, but they're still reported separately for the Delay/Anomaly page.

In [50]:
def compute_kpis(flights_c, fact):
    kpis = {}
    clean_flights = flights_c[~flights_c["is_anomaly"]]

    kpis["avg_duration_overall_min"] = clean_flights["duration_minutes"].mean()
    kpis["avg_duration_by_airline"] = clean_flights.groupby("airline")["duration_minutes"].mean().round(1).sort_values(ascending=False)
    kpis["avg_duration_by_route"] = clean_flights.groupby("route")["duration_minutes"].mean().round(1).sort_values(ascending=False)
    kpis["route_traffic"] = clean_flights["route"].value_counts()
    kpis["airline_distribution"] = flights_c["airline"].value_counts()
    kpis["overnight_share_pct"] = 100 * flights_c["is_overnight"].mean()
    kpis["anomaly_count"] = int(flights_c["is_anomaly"].sum())
    kpis["anomaly_flights"] = flights_c[flights_c["is_anomaly"]][
        ["flight_id", "airline", "route", "departure_time", "arrival_time", "duration_minutes"]]

    kpis["cancellation_rate_pct"] = 100 * (fact["status"] == "CANCELLED").mean()
    kpis["status_mix"] = fact["status"].value_counts()
    kpis["revenue_by_route"] = fact.groupby("route")["amount"].sum().round(2).sort_values(ascending=False)
    kpis["revenue_by_airline"] = fact.groupby("airline")["amount"].sum().round(2).sort_values(ascending=False)
    kpis["avg_fare_by_route"] = fact.groupby("route")["amount"].mean().round(2).sort_values(ascending=False)
    kpis["payment_method_mix"] = fact["payment_method"].value_counts()
    kpis["multi_attempt_payment_bookings"] = int((fact["payment_attempts"] > 1).sum())
    kpis["avg_passenger_age"] = fact["age"].mean()
    return kpis

kpis = compute_kpis(flights_c, fact)

print(f"Average flight duration (overall): {kpis['avg_duration_overall_min']:.1f} min")
print(f"Overnight (cross-day) flights: {kpis['overnight_share_pct']:.1f}%")
print(f"Anomalous flight records flagged: {kpis['anomaly_count']}")
print(f"Cancellation rate: {kpis['cancellation_rate_pct']:.1f}%")
print(f"Bookings with >1 payment attempt: {kpis['multi_attempt_payment_bookings']}")
print(f"Average passenger age: {kpis['avg_passenger_age']:.1f}")

Average flight duration (overall): 164.6 min
Overnight (cross-day) flights: 12.1%
Anomalous flight records flagged: 0
Cancellation rate: 31.4%
Bookings with >1 payment attempt: 267
Average passenger age: 42.9


In [51]:
print("Route-wise traffic (top 10):")
kpis["route_traffic"].head(10)

Route-wise traffic (top 10):


route
BOM-CCU    90
CCU-DEL    72
MAA-BLR    65
BLR-BOM    60
HYD-MAA    57
DEL-HYD    54
HYD-DEL    42
BOM-DEL    39
CCU-BOM    33
DEL-BLR    29
Name: count, dtype: int64

In [52]:
print("Distribution of flights by airline:")
kpis["airline_distribution"]

Distribution of flights by airline:


airline
IndiGo       273
Air India    255
SpiceJet     247
Vistara      230
Name: count, dtype: int64

In [53]:
print("Revenue by airline:")
kpis["revenue_by_airline"]

Revenue by airline:


airline
Vistara      2297985.04
SpiceJet     2025800.80
Air India    1902278.42
IndiGo       1779827.26
Name: amount, dtype: float64

## 9. Export cleaned tables + KPI CSVs (for Power BI and the report)

In [54]:
flights_c.to_csv(f"{CLEAN_DIR}/flights_clean.csv", index=False)
passengers_c.drop(columns=["first_name", "last_name", "email", "phone", "aadhaar_id", "date_of_birth"]).to_csv(f"{CLEAN_DIR}/passengers_clean.csv", index=False)
bookings_c.to_csv(f"{CLEAN_DIR}/bookings_clean.csv", index=False)
payments_c.to_csv(f"{CLEAN_DIR}/payments_clean.csv", index=False)

fact.to_csv(f"{OUT_DIR}/fact_bookings_powerbi.csv", index=False)
flights_c.to_csv(f"{OUT_DIR}/dim_flights_powerbi.csv", index=False)

for key, fname in [
    ("avg_duration_by_airline", "kpi_avg_duration_by_airline.csv"),
    ("avg_duration_by_route", "kpi_avg_duration_by_route.csv"),
    ("route_traffic", "kpi_route_traffic.csv"),
    ("airline_distribution", "kpi_airline_distribution.csv"),
    ("status_mix", "kpi_booking_status_mix.csv"),
    ("revenue_by_route", "kpi_revenue_by_route.csv"),
    ("revenue_by_airline", "kpi_revenue_by_airline.csv"),
    ("avg_fare_by_route", "kpi_avg_fare_by_route.csv"),
    ("payment_method_mix", "kpi_payment_method_mix.csv"),
]:
    kpis[key].to_csv(f"{OUT_DIR}/{fname}")
kpis["anomaly_flights"].to_csv(f"{OUT_DIR}/kpi_anomaly_flights.csv", index=False)

pd.DataFrame([{
    "avg_duration_overall_min": round(kpis["avg_duration_overall_min"], 1),
    "overnight_share_pct": round(kpis["overnight_share_pct"], 1),
    "anomaly_count": kpis["anomaly_count"],
    "cancellation_rate_pct": round(kpis["cancellation_rate_pct"], 1),
    "multi_attempt_payment_bookings": kpis["multi_attempt_payment_bookings"],
    "avg_passenger_age": round(kpis["avg_passenger_age"], 1),
    "total_bookings": len(fact),
    "total_revenue": round(fact["amount"].sum(), 2),
}]).to_csv(f"{OUT_DIR}/kpi_headline_summary.csv", index=False)

print("Done. Cleaned tables -> ./cleaned/   |   Power BI-ready tables + KPI CSVs -> ./output/")
print(sorted(os.listdir(OUT_DIR)))

Done. Cleaned tables -> ./cleaned/   |   Power BI-ready tables + KPI CSVs -> ./output/
['dim_flights_powerbi.csv', 'fact_bookings_powerbi.csv', 'kpi_airline_distribution.csv', 'kpi_anomaly_flights.csv', 'kpi_avg_duration_by_airline.csv', 'kpi_avg_duration_by_route.csv', 'kpi_avg_fare_by_route.csv', 'kpi_booking_status_mix.csv', 'kpi_headline_summary.csv', 'kpi_payment_method_mix.csv', 'kpi_revenue_by_airline.csv', 'kpi_revenue_by_route.csv', 'kpi_route_traffic.csv']
